In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..") / "src"))

### LLM


In [ ]:
import pandas as pd
from filters import corrosion_llm as corr

In [ ]:
corr.csv_to_jsonl_for_finetune(
    train_csv_path="../data/corrosion/corr_train.csv",
    jsonl_out_path="../data/corrosion/corr_train.jsonl",
)

In [ ]:
dataset = corr.load_jsonl("../data/corrosion/corr_train.jsonl")
errors = corr.validate_chat_jsonl_format(dataset)
errors

In [ ]:
API_KEY = "YOUR_KEY"  # or read from env, but keeping simple like your workflow

file_id = corr.upload_training_file(API_KEY, "../data/corrosion/corr_train.jsonl")
file_id

In [ ]:
job_id = corr.create_finetune_job(
    api_key=API_KEY,
    training_file_id=file_id,
    base_model="gpt-4o-2024-08-06",
)
job_id

In [ ]:
job_info = corr.get_finetune_job(API_KEY, job_id)
job_info["status"]

In [ ]:
# API_KEY = "YOUR_KEY"
FT_MODEL = "ft:gpt-4o-mini-2024-07-18:personal::BFBGbuz8"

df_pred = corr.predict_corrosion_for_csv(
    api_key=API_KEY,
    model_name=FT_MODEL,
    csv_path="../data/corrosion/corr_test.csv",
    smiles_col="smiles",
    out_csv_path="../data/corrosion/test_df_llm_preds.csv",
    system_prompt="You are a chemist assistant.",
    user_template="Is {smiles} corrosive to metals? Respond only with 0 for non-corrosive and 1 for corrosive.",
    temperature=0.0,
)

df_pred.head()

### Classical ML models

In [ ]:
from filters import corrosion_rf as crf

In [ ]:
rf_model, kept_cols = crf.train_rf_from_csv(
    train_csv_path="../data/corrosion/corr_train.csv",
    smiles_col="smiles",
    label_col="corrosion_status",
    n_estimators=100,
    random_state=42,
)

In [ ]:
df_with_preds, y_pred = crf.predict_rf_and_save(
    rf_model=rf_model,
    kept_columns=kept_cols,
    test_csv_path="../data/corrosion/corr_test.csv",
    output_csv_path="../data/corrosion/test_df_with_rf_preds.csv",
    smiles_col="smiles",
    pred_col_name="corr_pred",   # you can rename if you want
)